In [1]:
import pandas as pd
import numpy as np

In [2]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### The Network Name in db and Ted's spread

In [3]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")
df["Native ID"] = df["Native ID"].astype("string")
df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")

df_concat = df[df['ISSUE'].str.contains('Concatenate data and', case=False, na=False)]
df_concat = df_concat[['ISSUE','Station ID', 'Unique Names', 'Native ID', 'Network ID']].reset_index(drop=True)

# Split on '->'
split_ids = df_concat['Native ID'].astype(str).str.split('->', expand=True)

df_concat['old_native_id'] = split_ids[0].str.strip()
df_concat['new_native_id'] = split_ids[1].str.strip()
df_concat = df_concat.drop(columns=['Native ID'])

df_concat = df_concat.rename(columns={
    'Station ID': 'old_station_id',
    'Unique Names': 'old_station_name'
})

df_concat

,ISSUE,old_station_id,old_station_name,Network ID,old_native_id,new_native_id
0,Concatenate data and Delete this station,12627,NaN,36,14g,FW001
1,Concatenate data and Delete this station,12628,NaN,36,31n,FW003
2,Concatenate data and Delete this station,12629,NaN,36,4rw6,FW004
3,Concatenate data and Delete this station,12630,NaN,36,chris_creek,FW009
4,Concatenate data and Delete this station,12631,NaN,36,martins_gulch,FW007
5,Concatenate data and Delete this station,12632,NaN,36,mt_mcdonald,FW005
6,Concatenate data and Delete this station,12633,NaN,36,north_basin,FW008
7,Concatenate data and Delete this station,12634,NaN,36,sooke_dam,FW006
8,Concatenate data and Delete this station,12635,NaN,36,survey_mtn,HY031


In [4]:
select_sql = sa.text("""
SELECT DISTINCT
    n.network_id
FROM meta_station s
JOIN meta_network n
  ON n.network_id = s.network_id
JOIN meta_history m
  ON m.station_id = s.station_id
WHERE s.native_id = :native_id
""")

with engine.begin() as conn:
    for idx, row in df_concat.iterrows():

        native_id = row["new_native_id"]

        result = conn.execute(
            select_sql,
            {"native_id": native_id}
        ).fetchall()

        if not result:
            print(f"❌ native_id {native_id} not found")
            continue

        df_concat.at[idx, "new_network_id"] = ",".join(
              map(str, {r.network_id for r in result})
        )

df_concat

,ISSUE,old_station_id,old_station_name,Network ID,old_native_id,new_native_id,new_network_id
0,Concatenate data and Delete this station,12627,NaN,36,14g,FW001,36
1,Concatenate data and Delete this station,12628,NaN,36,31n,FW003,36
2,Concatenate data and Delete this station,12629,NaN,36,4rw6,FW004,36
3,Concatenate data and Delete this station,12630,NaN,36,chris_creek,FW009,36
4,Concatenate data and Delete this station,12631,NaN,36,martins_gulch,FW007,36
5,Concatenate data and Delete this station,12632,NaN,36,mt_mcdonald,FW005,36
6,Concatenate data and Delete this station,12633,NaN,36,north_basin,FW008,36
7,Concatenate data and Delete this station,12634,NaN,36,sooke_dam,FW006,36
8,Concatenate data and Delete this station,12635,NaN,36,survey_mtn,HY031,36


### check the obs records for both the old and new ID, then update the historical id in obs_raw, delete old station

In [5]:
query = """
SELECT
    -- old station native_id (from DB)
    (SELECT s.native_id
     FROM meta_station s
     WHERE s.station_id = %s
    ) AS old_native_id_db,

    -- old count (by old station_id)
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s) AS n_old,

    -- new count (by new native_id)
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s
     ) t) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND s_new.native_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(old_station_id, new_native_id, engine):

    params = (
        int(old_station_id),      # old_native_id_db
        int(old_station_id),      # n_old
        str(new_native_id),       # n_new
        int(old_station_id),      # n_overlap (old)
        str(new_native_id),       # n_overlap (new)
        int(old_station_id),      # n_overlap_same_datum (old)
        str(new_native_id)        # n_overlap_same_datum (new)
    )

    df = pd.read_sql(
        query,
        engine,
        params=params
    )

    return df.iloc[0]


stats = df_concat.apply(
    lambda r: count_station_stats(
        r['old_station_id'],
        r['new_native_id'],
        engine
    ),
    axis=1
)

df_concat[
    [
        'old_native_id_db',
        'n_old',
        'n_new',
        'n_overlap',
        'n_overlap_same_datum'
    ]
] = stats

df_concat

,ISSUE,old_station_id,old_station_name,Network ID,old_native_id,new_native_id,new_network_id,old_native_id_db,n_old,n_new,n_overlap,n_overlap_same_datum
0,Concatenate data and Delete this station,12627,NaN,36,14g,FW001,36,14g,1044863,3314861,61939,61939
1,Concatenate data and Delete this station,12628,NaN,36,31n,FW003,36,31n,960025,3075279,56215,56215
2,Concatenate data and Delete this station,12629,NaN,36,4rw6,FW004,36,4rw6,1081718,3470926,14873,14873
3,Concatenate data and Delete this station,12630,NaN,36,chris_creek,FW009,36,chris_creek,1045081,1144309,61886,61886
4,Concatenate data and Delete this station,12631,NaN,36,martins_gulch,FW007,36,martins_gulch,1050188,2137086,61885,61885
5,Concatenate data and Delete this station,12632,NaN,36,mt_mcdonald,FW005,36,mt_mcdonald,956746,1268159,33759,33759
6,Concatenate data and Delete this station,12633,NaN,36,north_basin,FW008,36,north_basin,999693,1299259,59063,59063
7,Concatenate data and Delete this station,12634,NaN,36,sooke_dam,FW006,36,sooke_dam,1069804,3204892,58458,47843
8,Concatenate data and Delete this station,12635,NaN,36,survey_mtn,HY031,36,survey_mtn,1097766,257302,59077,48468


In [6]:
df_concat

,ISSUE,old_station_id,old_station_name,Network ID,old_native_id,new_native_id,new_network_id,old_native_id_db,n_old,n_new,n_overlap,n_overlap_same_datum
0,Concatenate data and Delete this station,12627,NaN,36,14g,FW001,36,14g,1044863,3314861,61939,61939
1,Concatenate data and Delete this station,12628,NaN,36,31n,FW003,36,31n,960025,3075279,56215,56215
2,Concatenate data and Delete this station,12629,NaN,36,4rw6,FW004,36,4rw6,1081718,3470926,14873,14873
3,Concatenate data and Delete this station,12630,NaN,36,chris_creek,FW009,36,chris_creek,1045081,1144309,61886,61886
4,Concatenate data and Delete this station,12631,NaN,36,martins_gulch,FW007,36,martins_gulch,1050188,2137086,61885,61885
5,Concatenate data and Delete this station,12632,NaN,36,mt_mcdonald,FW005,36,mt_mcdonald,956746,1268159,33759,33759
6,Concatenate data and Delete this station,12633,NaN,36,north_basin,FW008,36,north_basin,999693,1299259,59063,59063
7,Concatenate data and Delete this station,12634,NaN,36,sooke_dam,FW006,36,sooke_dam,1069804,3204892,58458,47843
8,Concatenate data and Delete this station,12635,NaN,36,survey_mtn,HY031,36,survey_mtn,1097766,257302,59077,48468


### new speed code

In [7]:
from sqlalchemy import text

SQL_DELETE_DUPLICATES = text("""
DELETE FROM obs_raw o
USING obs_raw o2
WHERE o.vars_id = o2.vars_id
  AND o.obs_time = o2.obs_time
  AND o.history_id = :old_hist_id
  AND o2.history_id = :new_hist_id
""")

SQL_BULK_MOVE = text("""
UPDATE obs_raw
SET history_id = :new_hist_id
WHERE history_id = :old_hist_id
""")

SQL_GET_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
JOIN meta_station s ON h.station_id = s.station_id
WHERE s.native_id  = :native_id
  AND s.network_id = :network_id
ORDER BY h.history_id DESC
LIMIT 1
""")

def move_station_data_fast(old_native_id, new_native_id, old_network_id, new_network_id, engine):
    with engine.begin() as conn:

        old_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {
                "native_id": old_native_id,
                "network_id": old_network_id,
            }
        ).scalar()

        new_hist_id = conn.execute(
            SQL_GET_HISTORY,
            {
                "native_id": new_native_id,
                "network_id": new_network_id,
            }
        ).scalar()

        if old_hist_id is None or new_hist_id is None:
            print(f"❌ Missing history: {old_native_id} → {new_native_id}")
            return 0

        # 1️⃣ delete duplicates (fast)
        dup_res = conn.execute(
            SQL_DELETE_DUPLICATES,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        # 2️⃣ bulk reassign (very fast)
        move_res = conn.execute(
            SQL_BULK_MOVE,
            {
                "old_hist_id": old_hist_id,
                "new_hist_id": new_hist_id,
            }
        )

        print(
            f"    🧹 removed duplicates={dup_res.rowcount}, "
            f"🚚 moved={move_res.rowcount}"
        )

        return move_res.rowcount

results = []

for i, row in df_concat.iterrows():
    old_native_id = row["old_native_id"]
    new_native_id = row["new_native_id"]

    old_network_id = row["Network ID"]
    new_network_id = row["new_network_id"]


    print(f"[{i+1}/{len(df_concat)}] Moving {old_native_id} → {new_native_id}")

    n_moved = move_station_data_fast(old_native_id, new_native_id, old_network_id, new_network_id, engine)
    results.append(n_moved)

print("All done!")


[1/9] Moving 14g → FW001
    🧹 removed duplicates=61939, 🚚 moved=982924
[2/9] Moving 31n → FW003
    🧹 removed duplicates=56215, 🚚 moved=903810
[3/9] Moving 4rw6 → FW004
    🧹 removed duplicates=14873, 🚚 moved=1066845
[4/9] Moving chris_creek → FW009
    🧹 removed duplicates=61886, 🚚 moved=983195
[5/9] Moving martins_gulch → FW007
    🧹 removed duplicates=61885, 🚚 moved=988303
[6/9] Moving mt_mcdonald → FW005
    🧹 removed duplicates=33759, 🚚 moved=922987
[7/9] Moving north_basin → FW008
    🧹 removed duplicates=59063, 🚚 moved=940630
[8/9] Moving sooke_dam → FW006
    🧹 removed duplicates=58458, 🚚 moved=1011346
[9/9] Moving survey_mtn → HY031
    🧹 removed duplicates=59077, 🚚 moved=1038689
All done!


In [8]:
from tqdm import tqdm
import sqlalchemy as sa

station_ids = (
    df_concat["old_station_id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
# station_ids[0]


# List of station_ids to delete
station_ids_to_delete = station_ids  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations:   0%|          | 0/9 [00:00<?, ?it/s]

Deleting stations:  11%|█         | 1/9 [00:04<00:39,  4.89s/it]

Station 12627: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


Deleting stations:  22%|██▏       | 2/9 [00:05<00:18,  2.63s/it]

Station 12628: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


...

Station 12633: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


Deleting stations:  89%|████████▉ | 8/9 [00:15<00:01,  1.84s/it]

Station 12634: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1


Deleting stations: 100%|██████████| 9/9 [00:17<00:00,  1.91s/it]

Station 12635: flags=0, obs_raw=0, obs_derived=0, meta_history=1, meta_station=1
